In [1]:
import bz2
import json
import pandas as pd

from collections import Counter
from itertools import islice
from pathlib import Path

In [ ]:
PROJECT_ROOT = Path.cwd().parent
RAW_DIR = PROJECT_ROOT / "data" / "raw" / "wikidata"
WIKIDATA_PATH = RAW_DIR / "latest-all.json.bz2"

assert WIKIDATA_PATH.exists()

SAMPLE_SIZE = 1000

def iter_wikidata_entities(path: Path):
    with bz2.open(path, "rt", encoding="utf-8") as f:
        for raw_line in f:
            line = raw_line.strip()
            if not line or line in ("[", "]"):
                continue
            if line.endswith(","):
                line = line[:-1]
            yield json.loads(line)

def sample_entities(path: Path, limit: int = SAMPLE_SIZE):
    return list(islice(iter_wikidata_entities(path), limit))

def summarize_entity(entity: dict) -> dict:
    return {
        "id": entity.get("id"),
        "type": entity.get("type"),
        "modified": entity.get("modified"),
        "lastrevid": entity.get("lastrevid"),
        "label_langs": len(entity.get("labels", {})),
        "alias_langs": len(entity.get("aliases", {})),
        "description_langs": len(entity.get("descriptions", {})),
        "claim_properties": len(entity.get("claims", {})),
        "sitelinks": len(entity.get("sitelinks", {})),
    }

def flatten_labels(entities: list[dict]) -> pd.DataFrame:
    rows = []
    for entity in entities:
        for lang, value in entity.get("labels", {}).items():
            rows.append(
                {
                    "id": entity.get("id"),
                    "entity_type": entity.get("type"),
                    "lang": lang,
                    "label": value.get("value"),
                }
            )
    return pd.DataFrame(rows)

def flatten_aliases(entities: list[dict]) -> pd.DataFrame:
    rows = []
    for entity in entities:
        for lang, aliases in entity.get("aliases", {}).items():
            for alias in aliases:
                rows.append(
                    {
                        "id": entity.get("id"),
                        "entity_type": entity.get("type"),
                        "lang": lang,
                        "alias": alias.get("value"),
                    }
                )
    return pd.DataFrame(rows)

def flatten_claim_counts(entities: list[dict]) -> pd.DataFrame:
    counter = Counter()
    for entity in entities:
        for prop, claims in entity.get("claims", {}).items():
            counter[prop] += len(claims)
    return (
        pd.DataFrame(
            [{"property": prop, "claims_in_sample": count} for prop, count in counter.items()]
        )
        .sort_values("claims_in_sample", ascending=False)
        .reset_index(drop=True)
    )

In [ ]:
entities = sample_entities(WIKIDATA_PATH, SAMPLE_SIZE)

print(f"Sampled entities: {len(entities)}")

Using dump: latest-all.json.bz2
Sampled entities: 1000


In [4]:
df_entities = pd.DataFrame(summarize_entity(entity) for entity in entities)
df_entities.head()

,id,type,modified,lastrevid,label_langs,alias_langs,description_langs,claim_properties,sitelinks
0,Q31,item,2026-04-11T15:44:14Z,2480332924,333,101,127,390,372
1,Q8,item,2026-04-06T23:49:39Z,2478991506,172,82,83,82,171
2,Q23,item,2026-04-10T19:10:12Z,2480129897,111,48,107,321,269
3,Q24,item,2026-02-04T02:15:55Z,2460497146,41,4,53,41,26
4,Q42,item,2026-04-11T22:30:28Z,2480467219,77,25,119,297,132


In [5]:
df_labels = flatten_labels(entities)
df_labels.head()

,id,entity_type,lang,label
0,Q31,item,el,Βέλγιο
1,Q31,item,ay,Bilkiya
2,Q31,item,pnb,بیلجیئم
3,Q31,item,na,Berdjiyum
4,Q31,item,mk,Белгија


In [6]:
df_aliases = flatten_aliases(entities)
df_aliases.head()

,id,entity_type,lang,alias
0,Q31,item,bg,Кралство Белгия
1,Q31,item,mk,Кралство Белгија
2,Q31,item,zh-hk,比利時王國
3,Q31,item,pt-br,Reino da Bélgica
4,Q31,item,fr,Royaume de Belgique


In [7]:
df_claim_counts = flatten_claim_counts(entities)

pd.DataFrame(
    {
        "entity_type": df_entities["type"].value_counts().index,
        "count": df_entities["type"].value_counts().values,
    }
)

,entity_type,count
0,item,1000


In [8]:
top_label_langs = (
    df_labels["lang"].value_counts().rename_axis("lang").reset_index(name="label_count")
)

top_alias_langs = (
    df_aliases["lang"].value_counts().rename_axis("lang").reset_index(name="alias_count")
    if not df_aliases.empty
    else pd.DataFrame(columns=["lang", "alias_count"])
)

print("Top label languages")
display(top_label_langs.head(20))

print("\nTop alias languages")
display(top_alias_langs.head(20))

print("\nTop claim properties in sample")
display(df_claim_counts.head(20))

Top label languages


,lang,label_count
0,en,996
1,de,992
2,es,989
3,fr,989
4,it,986
5,nl,962
6,ru,942
7,ca,918
8,ja,878
9,zh,875



Top alias languages


,lang,alias_count
0,en,2190
1,de,920
2,es,840
3,ar,798
4,fr,756
5,zh,750
6,ru,669
7,ja,546
8,it,495
9,cs,417



Top claim properties in sample


,property,claims_in_sample
0,P1082,3153
1,P150,2925
2,P1343,1931
3,P31,1672
4,P2936,1352
5,P47,1316
6,P1549,1082
7,P974,1068
8,P527,996
9,P190,980
